# Simple SMA Strategy — Backtest

A **trend-following** strategy built from three Simple Moving Averages:

- a **fast** MA and a **slow** MA whose **crossover** triggers entries
  (fast crossing above slow → go long; below → go short);
- a third, **longer trend** MA that acts as a **regime filter** — we only take
  longs while price is above it, shorts while below. This keeps us on the side of
  the dominant trend.

Exits are a fixed take-profit / stop-loss set at entry (`+30%` / `-15%` for longs)
or the reverse crossover. Like the envelope notebook, this is *event-driven*: the
backtester steps through candles one bar at a time on the Polars-native
`quant_research` library.

In [ ]:
import polars as pl

from quant_research.connectors import CCXTLoader          # load cached candles
from quant_research.strategies import SimpleSMAStrategy    # the strategy logic
from quant_research.backtest import BacktestAnalysis       # metrics + charts

## Step 1 — Load the price data

Daily BTC spot candles from the cache (run `data_engine.ipynb` first if empty).
Daily bars suit a slow trend system with 100/200/300-period SMAs.

In [ ]:
loader = CCXTLoader(exchange="binance")
symbol = "BTC/USDT"                       # spot pair
ohlcv = loader.load(symbol, timeframe="1d")   # daily candles

## Step 2 — Configure and run

The three SMA periods define the system; `position_size_percentage` sets how much
of the wallet each trade uses. `run_backtest` walks the candles bar by bar with a
starting balance, leverage and a per-side `fee_rate` (0.0006 = 6 bps).

Take-profit / stop-loss are fixed ratios baked into the strategy class
(`+30%` / `-15%` for longs, mirrored for shorts).

In [ ]:
strategy_params = {
    "fast_ma_period": 100,             # fast SMA — reacts quickest
    "slow_ma_period": 200,             # slow SMA — the crossover partner
    "trend_ma_period": 300,            # trend filter — only trade with this regime
    "position_size_percentage": 100,   # use 100% of wallet per trade
    # "position_size_fixed_amount": 100,   # alt: fixed quote amount
    # "position_size_exposure": 2,         # alt: exposure multiple
    # "mode": "long" | "short" | "both"    # default 'both'
}
strategy = SimpleSMAStrategy(strategy_params, ohlcv)
strategy.run_backtest(
    initial_balance=1000,   # start with 1000 USDT
    leverage=1,             # no leverage
    fee_rate=0.0006,        # 6 bps per side
)

## Step 3 — Performance metrics

In [ ]:
results = BacktestAnalysis(strategy)
results.print_metrics()

## Step 4 — Visualize

In [ ]:
results.plot_equity()

In [ ]:
results.plot_drawdown()

In [ ]:
results.plot_monthly_performance()

## Step 5 — Candlestick with the three SMAs

Overlay the fast / slow / trend lines on the price so you can see the crossovers
and which side of the trend filter price was on. Same `indicators` format as the
envelope notebook: **name → {color, df(time, value)}**.

In [ ]:
indicators = {
    "fastMA": {"color": "gold",   "df": strategy.data.select(pl.col("datetime").alias("time"), pl.col("fastMA")).drop_nulls()},
    "slowMA": {"color": "purple", "df": strategy.data.select(pl.col("datetime").alias("time"), pl.col("slowMA")).drop_nulls()},
    "trend":  {"color": "white",  "df": strategy.data.select(pl.col("datetime").alias("time"), pl.col("trend")).drop_nulls()},
}
results.plot_candlestick(indicators=indicators)

## Exercises

1. **Speed** — try faster SMAs (e.g. 20/50/100). More trades, more noise — does
   the net result improve after fees?
2. **Filter** — remove the trend filter idea by setting `trend_ma_period` very
   small. How much does the regime filter actually help?
3. **Sizing** — switch to `position_size_exposure` or a fixed amount and compare
   the equity curve.
4. **Direction** — restrict to `"mode": "long"` (crypto spends long stretches
   trending up) and compare.